# MuscleMap Thigh — Sheffield Dataset Evaluation

Computes per-muscle Dice / Hausdorff metrics for MuscleMap **thigh** segmentations on the **Sheffield** dataset.

- **Predictions**: `../sheffield_segs/Aug_N_dseg.nii.gz` (labels 1–28, thigh model Zenodo 19633000)
- **Ground truth**: `../../sheffeld/20440203/Aug_N_segmentations.dcm` (labels 1–37, bilateral)

Sheffield GT labels are **bilateral** — L and R model labels are OR-combined before comparison.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'pydicom', 'pandas', 'numpy'])

import numpy as np

# ── Inlined from dissector.evaluation (no local package install needed) ────────

def binary_cross_entropy(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-7) -> float:
    y_pred_clipped = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred_clipped) + (1 - y_true) * np.log(1 - y_pred_clipped))


def inter_slice_dice(mask: np.ndarray) -> float:
    scores = []
    for i in range(mask.shape[0] - 1):
        a = mask[i].astype(bool)
        b = mask[i + 1].astype(bool)
        denom = a.sum() + b.sum()
        if denom == 0:
            continue
        scores.append(2 * np.logical_and(a, b).sum() / denom)
    return float(np.mean(scores)) if scores else 0.0


def _extract_boundary_3d(mask: np.ndarray) -> np.ndarray:
    d, h, w = mask.shape
    boundary = np.zeros_like(mask, dtype=bool)
    for z in range(d):
        for i in range(h):
            for j in range(w):
                if not mask[z, i, j]:
                    continue
                for dz, di, dj in [(-1,0,0),(1,0,0),(0,-1,0),(0,1,0),(0,0,-1),(0,0,1)]:
                    nz, ni, nj = z+dz, i+di, j+dj
                    if (nz < 0 or nz >= d or ni < 0 or ni >= h or
                            nj < 0 or nj >= w or not mask[nz, ni, nj]):
                        boundary[z, i, j] = True
                        break
    return boundary


def _dilate_3d(mask: np.ndarray, distance: int) -> np.ndarray:
    h, w, d = mask.shape
    dilated = np.zeros_like(mask, dtype=bool)
    for i in range(h):
        for j in range(w):
            for k in range(d):
                if not mask[i, j, k]:
                    continue
                for di in range(-distance, distance + 1):
                    for dj in range(-distance, distance + 1):
                        for dk in range(-distance, distance + 1):
                            ni, nj, nk = i+di, j+dj, k+dk
                            if 0 <= ni < h and 0 <= nj < w and 0 <= nk < d:
                                dilated[ni, nj, nk] = True
    return dilated


def boundary_iou_3d(distance: int, mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    b_a = _extract_boundary_3d(mask_a)
    b_b = _extract_boundary_3d(mask_b)
    d_a = _dilate_3d(b_a, distance)
    d_b = _dilate_3d(b_b, distance)
    intersection = np.logical_and(d_a, d_b).sum()
    union = d_a.sum() + d_b.sum() - intersection
    return float(intersection / union) if union else 0.0

print('Dependencies and evaluation functions ready')

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom

In [ ]:
BOUNDARY_DISTANCE = 1
GT_DIR     = os.path.join('..', '..', 'sheffeld', '20440203')
SEG_DIR    = os.path.join('..', 'sheffield_segs')
RESULT_DIR = 'results_sheffield'
ALGO_TAG   = 'musclemap_thigh'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, sheffield_gt_label, mm_thigh_labels)
# MM thigh 1-28 label scheme (Zenodo 19633000): L labels are odd, R labels are even.
# Sheffield GT is bilateral — OR both sides.
MUSCLES = [
    ('adductor_brevis',    1,  [25, 26]),
    ('adductor_longus',    2,  [23, 24]),
    ('adductor_magnus',    3,  [21, 22]),
    ('biceps_femoris_short', 4, [19, 20]),
    ('biceps_femoris_long',  5, [17, 18]),
    ('gracilis',           16, [11, 12]),
    ('rectus_femoris',     27, [7,  8 ]),
    ('sartorius',          28, [9,  10]),
    ('semimembranosus',    29, [13, 14]),
    ('semitendinosus',     30, [15, 16]),
    ('vastus_intermedius', 35, [3,  4 ]),
    ('vastus_lateralis',   36, [1,  2 ]),
    ('vastus_medialis',    37, [5,  6 ]),
]

def read_gt(idx):
    ds  = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    raw = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)

def get_spacing(idx):
    ds = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    ps = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st = float(getattr(ds, 'SliceThickness', 1.0))
    return (float(ps[1]), float(ps[0]), st)

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, 'Aug_*_dseg.nii.gz')),
                   key=lambda p: int(re.search(r'Aug_(\d+)', p).group(1)))
print(f'GT_DIR  : {os.path.abspath(GT_DIR)}')
print(f'SEG_DIR : {os.path.abspath(SEG_DIR)}')
print(f'Found   : {len(seg_files)} NIfTI files')

In [ ]:
def evaluate_muscle(muscle_name, sheffield_label, pred_labels):
    results = []
    for seg_path in seg_files:
        idx = re.search(r'Aug_(\d+)', seg_path.replace('\\', '/')).group(1)
        gt_path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: Aug_{idx}'); continue
        gt_arr  = read_gt(idx)
        spacing = get_spacing(idx)
        gt_bin  = (gt_arr == sheffield_label).astype(np.uint8)

        pred_sitk = sitk.ReadImage(seg_path)
        seg_raw   = sitk.GetArrayFromImage(pred_sitk)
        if seg_raw.shape != gt_arr.shape:
            ref = sitk.GetImageFromArray(gt_arr.astype(np.int32)); ref.SetSpacing(spacing)
            pred_sitk = sitk.Resample(pred_sitk, ref, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
            seg_raw   = sitk.GetArrayFromImage(pred_sitk)
        pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)
        for lbl in pred_labels:
            pred_arr |= (seg_raw == lbl).astype(np.uint8)

        gt_sitk2  = sitk.GetImageFromArray(gt_bin);   gt_sitk2.SetSpacing(spacing)
        pred_sitk2= sitk.GetImageFromArray(pred_arr); pred_sitk2.SetSpacing(spacing)
        dice_f = sitk.LabelOverlapMeasuresImageFilter(); dice_f.Execute(gt_sitk2, pred_sitk2)
        if gt_bin.sum() > 0 and pred_arr.sum() > 0:
            hd_f = sitk.HausdorffDistanceImageFilter(); hd_f.Execute(gt_sitk2, pred_sitk2)
            hd = hd_f.GetHausdorffDistance()
        else:
            hd = np.nan
        results.append({
            'sample': f'Aug_{idx}',
            f'{muscle_name}_dice':                  dice_f.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_f.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_f.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_f.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_f.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_bin.astype(float), pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_bin.astype(float)),
        })
    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_sheffield.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

dfs = {}
for muscle_name, sheffield_label, pred_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={sheffield_label}, MM={pred_labels}) ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, sheffield_label, pred_labels)
print('\nDone.')

In [ ]:
from IPython.display import display
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty: continue
    summary_rows.append({
        'muscle': muscle_name, 'n': len(df),
        'dice_mean': df[f'{muscle_name}_dice'].mean(), 'dice_std': df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(), 'hausdorff_std': df[f'{muscle_name}_hausdorff'].std(),
    })
summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_sheffield.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))